# data_understanding_historical_risk_patch

Notebook hasil konversi dari `data_understanding_historical_risk_patch.py`. Kode dipisah ke beberapa cell berdasarkan blok top-level agar lebih mudah dibaca, dijalankan, dan di-screenshot.


## Imports


In [ ]:
from __future__ import annotations

import argparse
import json
import re
from dataclasses import asdict, dataclass
from datetime import datetime
from functools import lru_cache
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter


## Konstanta dan Setup Awal


In [ ]:
DATE_PATTERN = re.compile(r"FIRMS_(\d{4}-\d{2}-\d{2})")
DEFAULT_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


## Class: `DatasetRecord`


In [ ]:
class DatasetRecord:
    path: Path
    date: datetime


## Function: `parse_image_extensions`


In [ ]:
def parse_image_extensions(value: str) -> tuple[str, ...]:
    extensions: list[str] = []
    for item in value.split(","):
        ext = item.strip().lower()
        if not ext:
            continue
        if not ext.startswith("."):
            ext = f".{ext}"
        if ext not in extensions:
            extensions.append(ext)
    return tuple(extensions) if extensions else DEFAULT_IMAGE_EXTENSIONS


## Function: `load_records`


In [ ]:
def load_records(dataset_dir: Path, image_extensions: tuple[str, ...]) -> list[DatasetRecord]:
    records: list[DatasetRecord] = []
    allowed_extensions = set(image_extensions)
    for path in sorted(item for item in dataset_dir.iterdir() if item.is_file()):
        if path.suffix.lower() not in allowed_extensions:
            continue
        match = DATE_PATTERN.search(path.name)
        if not match:
            continue
        records.append(DatasetRecord(path=path, date=datetime.strptime(match.group(1), "%Y-%m-%d")))
    return sorted(records, key=lambda item: item.date)


## Function: `validate_records`


In [ ]:
def validate_records(records: list[DatasetRecord]) -> tuple[list[DatasetRecord], list[dict[str, str]]]:
    valid_records: list[DatasetRecord] = []
    skipped_records: list[dict[str, str]] = []
    for record in records:
        try:
            if not record.path.exists():
                raise FileNotFoundError(f"File tidak ditemukan: {record.path}")
            with Image.open(record.path) as image:
                image.verify()
            valid_records.append(record)
        except Exception as exc:
            skipped_records.append({"path": str(record.path), "reason": str(exc)})
    return valid_records, skipped_records


## Function: `build_sample_starts`


In [ ]:
def build_sample_starts(record_count: int, seq_length: int) -> list[int]:
    return list(range(record_count - seq_length))


## Function: `split_sample_starts`


In [ ]:
def split_sample_starts(
    sample_starts: list[int],
    train_ratio: float,
    val_ratio: float,
) -> tuple[list[int], list[int], list[int]]:
    sample_count = len(sample_starts)
    train_end = max(1, int(sample_count * train_ratio))
    val_end = max(train_end + 1, int(sample_count * (train_ratio + val_ratio)))
    val_end = min(val_end, sample_count - 1)
    train = sample_starts[:train_end]
    val = sample_starts[train_end:val_end]
    test = sample_starts[val_end:]
    if not val or not test:
        raise ValueError("Split train/val/test gagal. Periksa rasio dataset.")
    return train, val, test


## Function: `_normalize_kernel`


In [ ]:
def _normalize_kernel(size: int) -> int:
    size = max(1, int(size))
    return size if size % 2 == 1 else size + 1


## Function: `load_native_risk_map`


In [ ]:
def load_native_risk_map(
    path_str: str,
    native_width: int,
    native_height: int,
    label_dilation_kernel: int,
    label_blur_radius: float,
) -> np.ndarray:
    label_dilation_kernel = _normalize_kernel(label_dilation_kernel)
    with Image.open(path_str) as image:
        rgb = image.convert("RGB")
        hsv = np.asarray(rgb.convert("HSV"), dtype=np.uint8)

    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]
    red_low = (h <= 14) & (s >= 70) & (v >= 50)
    red_high = (h >= 242) & (s >= 70) & (v >= 50)
    mask = ((red_low | red_high).astype(np.uint8)) * 255

    risk_image = Image.fromarray(mask)
    if label_dilation_kernel > 1:
        risk_image = risk_image.filter(ImageFilter.MaxFilter(size=label_dilation_kernel))
    if label_blur_radius > 0:
        risk_image = risk_image.filter(ImageFilter.GaussianBlur(radius=float(label_blur_radius)))
    if risk_image.size != (native_width, native_height):
        risk_image = risk_image.resize((native_width, native_height), Image.BILINEAR)

    density = np.asarray(risk_image, dtype=np.float32) / 255.0
    return np.clip(density, 0.0, 1.0)


## Function: `estimate_positive_weight`


In [ ]:
def estimate_positive_weight(
    records: list[DatasetRecord],
    sample_starts: list[int],
    seq_length: int,
    native_width: int,
    native_height: int,
    label_dilation_kernel: int,
    label_blur_radius: float,
    max_pos_weight: float,
) -> tuple[float, float]:
    positive_sum = 0.0
    pixel_count = 0
    for start in sample_starts:
        mask = load_native_risk_map(
            str(records[start + seq_length].path),
            native_width,
            native_height,
            label_dilation_kernel,
            label_blur_radius,
        )
        positive_sum += float(mask.sum())
        pixel_count += mask.size
    positive_ratio = max(positive_sum / max(pixel_count, 1), 1e-6)
    pos_weight = min((1.0 - positive_ratio) / positive_ratio, max_pos_weight)
    return float(max(pos_weight, 1.0)), float(positive_ratio)


## Function: `build_parser`


In [ ]:
def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=(
            "Ringkasan Data Understanding untuk historical risk patch. "
            "Script ini hanya membaca dataset, split kronologis, dan opsional menghitung class imbalance."
        )
    )
    parser.add_argument(
        "--dataset-dir",
        default="Ipynb/Dataset History Fire Hotspot In Riau Province PNG",
        help="Folder dataset gambar hotspot historis.",
    )
    parser.add_argument(
        "--image-extensions",
        default=".png",
        help="Daftar ekstensi gambar, pisahkan dengan koma. Contoh: .jpg,.jpeg,.png",
    )
    parser.add_argument("--seq-length", type=int, default=7)
    parser.add_argument("--train-ratio", type=float, default=0.70)
    parser.add_argument("--val-ratio", type=float, default=0.15)
    parser.add_argument("--native-width", type=int, default=1528)
    parser.add_argument("--native-height", type=int, default=773)
    parser.add_argument("--label-dilation-kernel", type=int, default=9)
    parser.add_argument("--label-blur-radius", type=float, default=2.0)
    parser.add_argument("--max-pos-weight", type=float, default=50.0)
    parser.add_argument(
        "--with-positive-weight",
        action="store_true",
        help="Hitung positive mass ratio dan positive class weight seperti training script utama.",
    )
    parser.add_argument(
        "--show-skipped",
        action="store_true",
        help="Tampilkan daftar file yang dilewati saat validasi.",
    )
    parser.add_argument(
        "--json",
        action="store_true",
        help="Cetak ringkasan akhir dalam format JSON.",
    )
    return parser


## Function: `build_summary`


In [ ]:
def build_summary(args: argparse.Namespace) -> dict:
    dataset_dir = Path(args.dataset_dir).expanduser().resolve()
    if not dataset_dir.exists():
        raise FileNotFoundError(f"Dataset tidak ditemukan: {dataset_dir}")

    image_extensions = parse_image_extensions(args.image_extensions)
    records = load_records(dataset_dir, image_extensions)
    if len(records) <= args.seq_length:
        raise ValueError("Jumlah data tidak cukup untuk membentuk sequence.")

    valid_records, skipped_records = validate_records(records)
    if len(valid_records) <= args.seq_length:
        raise ValueError("Jumlah data valid tidak cukup untuk membentuk sequence.")

    sample_starts = build_sample_starts(len(valid_records), args.seq_length)
    train_starts, val_starts, test_starts = split_sample_starts(sample_starts, args.train_ratio, args.val_ratio)
    pos_weight: float | None = None
    positive_ratio: float | None = None
    if args.with_positive_weight:
        pos_weight, positive_ratio = estimate_positive_weight(
            valid_records,
            train_starts,
            args.seq_length,
            args.native_width,
            args.native_height,
            args.label_dilation_kernel,
            args.label_blur_radius,
            args.max_pos_weight,
        )

    return {
        "dataset_dir": str(dataset_dir),
        "image_extensions": list(image_extensions),
        "record_count_total": len(records),
        "record_count_valid": len(valid_records),
        "record_count_skipped": len(skipped_records),
        "date_start": valid_records[0].date.date().isoformat(),
        "date_end": valid_records[-1].date.date().isoformat(),
        "seq_length": args.seq_length,
        "sample_count_total": len(sample_starts),
        "train_samples": len(train_starts),
        "val_samples": len(val_starts),
        "test_samples": len(test_starts),
        "positive_mass_ratio_train": positive_ratio,
        "positive_class_weight": pos_weight,
        "positive_weight_computed": args.with_positive_weight,
        "train_ratio": args.train_ratio,
        "val_ratio": args.val_ratio,
        "test_ratio_effective": len(test_starts) / max(len(sample_starts), 1),
        "skipped_records_preview": skipped_records[:10],
        "first_record": asdict(valid_records[0]) | {"path": str(valid_records[0].path)},
        "last_record": asdict(valid_records[-1]) | {"path": str(valid_records[-1].path)},
    }


## Function: `print_human_summary`


In [ ]:
def print_human_summary(summary: dict, show_skipped: bool) -> None:
    print("[data_understanding] Dataset ditemukan:", f"{summary['record_count_valid']} frame valid")
    print("[data_understanding] Total file cocok ekstensi:", summary["record_count_total"])
    print("[data_understanding] Rentang data:", f"{summary['date_start']} s.d. {summary['date_end']}")
    print("[data_understanding] Seq length:", summary["seq_length"])
    print("[data_understanding] Total sample sequence:", summary["sample_count_total"])
    print("[data_understanding] Train samples:", summary["train_samples"])
    print("[data_understanding] Val samples:", summary["val_samples"])
    print("[data_understanding] Test samples:", summary["test_samples"])
    if summary["positive_weight_computed"]:
        print(
            "[data_understanding] Positive mass ratio (train):",
            f"{summary['positive_mass_ratio_train']:.8f}",
        )
        print(
            "[data_understanding] Positive class weight:",
            f"{summary['positive_class_weight']:.4f}",
        )
    else:
        print("[data_understanding] Positive mass ratio (train): dilewati")
        print("[data_understanding] Positive class weight: dilewati")
        print("[data_understanding] Tips: pakai --with-positive-weight untuk menghitung nilai exact.")

    if summary["record_count_skipped"] > 0:
        print("[data_understanding] File dilewati:", summary["record_count_skipped"])
        if show_skipped:
            for item in summary["skipped_records_preview"]:
                print(f"- skip {item['path']} | alasan: {item['reason']}")


## Function: `main`


In [ ]:
def main() -> None:
    parser = build_parser()
    args = parser.parse_args()
    summary = build_summary(args)
    print_human_summary(summary, show_skipped=args.show_skipped)
    if args.json:
        print()
        print(json.dumps(summary, indent=2, default=str))


## Entry Point


In [ ]:
if __name__ == "__main__":
    main()
